In [ ]:
!pip -q install transformers datasets jiwer sentencepiece accelerate librosa soundfile safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa

from tqdm.auto import tqdm
from jiwer import wer, cer

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===============================
# INPUT DATA
# ===============================
INPUT_CSV = "/content/drive/MyDrive/dataset_corrected_juniors.csv"
SAMPLED_CSV = "/content/drive/MyDrive/dataset_corrected_juniors_sample_1000_hint.csv"

# ===============================
# WHISPER
# ===============================
MERGED_MODEL_PATH = "/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment4/merged_full_model"

# ===============================
# XLM-R LARGE DUAL-HEAD
# ===============================
XLMR_OUTPUT_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2"
XLMR_MODEL_DIR = os.path.join(XLMR_OUTPUT_DIR, "best_model")
ROOT_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2root.json")
SUFFIX_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2suffix.json")
TRAIN_DATA_PATH = "/content/drive/MyDrive/dual_head_training_data.csv"

# ===============================
# mT5 MERGED MODEL
# ===============================
MT5_MODEL_PATH = "/content/drive/MyDrive/mt5_merged_model"

# ===============================
# OUTPUTS
# ===============================
ASR_RESULTS_CSV = "/content/drive/MyDrive/hint_pipeline_asr_results_1000.csv"
HINT_RESULTS_CSV = "/content/drive/MyDrive/hint_pipeline_with_hints_1000.csv"
FINAL_RESULTS_CSV = "/content/drive/MyDrive/hint_pipeline_mt5_results_1000.csv"
FINAL_METRICS_JSON = "/content/drive/MyDrive/hint_pipeline_metrics_1000.json"
DEBUG_HINTS_CSV = "/content/drive/MyDrive/hint_pipeline_debug_hints_1000.csv"

# ===============================
# GENERAL
# ===============================
SAMPLE_SIZE = 1000
RANDOM_SEED = 42
TARGET_SR = 16000
MAX_LEN_XLMR = 96

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

Device: cuda


In [ ]:
df = pd.read_csv(INPUT_CSV)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))

required_cols = ["id", "script_text", "audio_wav_path", "duration"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
sample_df.to_csv(SAMPLED_CSV, index=False)

print("Sampled rows:", len(sample_df))
print("Saved sampled CSV:", SAMPLED_CSV)
sample_df.head()

Columns: ['id', 'script_text', 'audio_wav_path', 'duration']
Total rows: 25897
Sampled rows: 1000
Saved sampled CSV: /content/drive/MyDrive/dataset_corrected_juniors_sample_1000_hint.csv


,id,script_text,audio_wav_path,duration
0,8a13434a-3693-40d9-986a-a231316ecda2,Online class join பண்ண late ஆனதால start miss ஆ...,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14
2,fd532190-fb3d-4bce-9139-c1cc6a644601,Etsy help center site-ல selling on etsy-ல paym...,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86


In [ ]:
processor = WhisperProcessor.from_pretrained(MERGED_MODEL_PATH)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    MERGED_MODEL_PATH,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

if device != "cuda":
    whisper_model = whisper_model.to(device)

whisper_model.eval()
whisper_model.generation_config.max_length = None

print("Whisper model loaded.")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Whisper model loaded.


In [ ]:
def load_audio(audio_path, target_sr=16000):
    audio, sr = librosa.load(audio_path, sr=target_sr)
    return audio

def clean_whisper_text(text):
    text = re.sub(r"<\|.*?\|>", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def transcribe_audio_whisper(audio_path):
    try:
        audio = load_audio(audio_path, TARGET_SR)

        inputs = processor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16 if device == "cuda" else torch.float32
        )

        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                input_features,
                max_new_tokens=225
            )

        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        text = clean_whisper_text(text)
        return text.strip(), None

    except Exception as e:
        return "", str(e)

In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()

whisper_model.generation_config.max_length = None

asr_rows = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Running Whisper ASR"):
    generated_text, error_msg = transcribe_audio_whisper(str(row["audio_wav_path"]).strip())

    asr_rows.append({
        "id": row["id"],
        "audio_wav_path": str(row["audio_wav_path"]).strip(),
        "duration": row["duration"],
        "expected_text": str(row["script_text"]).strip(),
        "generated_text": generated_text,
        "error": error_msg
    })

asr_df = pd.DataFrame(asr_rows)
asr_df.to_csv(ASR_RESULTS_CSV, index=False)

print("Saved ASR results:", ASR_RESULTS_CSV)
asr_df.head()

Running Whisper ASR:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved ASR results: /content/drive/MyDrive/hint_pipeline_asr_results_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [ ]:
def clean_token(tok):
    tok = str(tok).strip()
    tok = tok.strip(".,!?;:\"“”‘’()[]{}")
    return tok

def tokenize(text):
    return [clean_token(tok) for tok in str(text).strip().split() if clean_token(tok)]

def is_english_word(token):
    return bool(re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", token))

def is_tamil_text(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", text))

def is_mixed_token(token):
    if "-" not in token:
        return False
    parts = token.split("-", 1)
    if len(parts) != 2:
        return False
    left, right = parts[0].strip(), parts[1].strip()
    return is_english_word(left) and is_tamil_text(right)

def get_token_class(token):
    if is_mixed_token(token):
        return "MIX"
    elif is_english_word(token):
        return "EN"
    elif is_tamil_text(token):
        return "TA"
    else:
        return "OTHER"

def split_mixed_token(token):
    if "-" not in token:
        return None, None
    left, right = token.split("-", 1)
    return left.strip(), right.strip()

def extract_root_suffix(token, token_class):
    if token_class == "MIX":
        root, suffix = split_mixed_token(token)
        return root if root else "", suffix if suffix else ""
    elif token_class == "EN":
        return token, "NULL"
    return "", ""

def build_model_input(left_context, token, right_context, token_class, root, suffix):
    return f"LEFT={left_context} TOKEN={token} RIGHT={right_context} CLASS={token_class} ROOT={root} SUFFIX={suffix}"

def get_context(tokens, idx, window=2):
    left_tokens = tokens[max(0, idx-window):idx]
    right_tokens = tokens[idx+1:idx+1+window]
    return " ".join(left_tokens).strip(), " ".join(right_tokens).strip()

In [ ]:
with open(ROOT_MAP_PATH, "r", encoding="utf-8") as f:
    id2root = json.load(f)
with open(SUFFIX_MAP_PATH, "r", encoding="utf-8") as f:
    id2suffix = json.load(f)

id2root = {int(k): v for k, v in id2root.items()}
id2suffix = {int(k): v for k, v in id2suffix.items()}

train_df = pd.read_csv(TRAIN_DATA_PATH)
changed_df = train_df[train_df["generated_token"] != train_df["expected_token"]].copy()

from collections import Counter
token_change_counter = Counter(changed_df["generated_token"].astype(str).tolist())

SUSPICIOUS_TOKEN_MIN_COUNT = 2
suspicious_tokens = {
    tok for tok, cnt in token_change_counter.items()
    if cnt >= SUSPICIOUS_TOKEN_MIN_COUNT
}

print("Root labels:", len(id2root))
print("Suffix labels:", len(id2suffix))
print("Suspicious tokens:", len(suspicious_tokens))

Root labels: 796
Suffix labels: 62
Suspicious tokens: 1803


In [ ]:
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        return {
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }

xlmr_model = XLMRDualHeadModel(
    model_name="xlm-roberta-large",
    num_root_labels=len(id2root),
    num_suffix_labels=len(id2suffix)
)

state_dict_path = os.path.join(XLMR_MODEL_DIR, "model.safetensors")

if os.path.exists(state_dict_path):
    from safetensors.torch import load_file
    state_dict = load_file(state_dict_path)
    xlmr_model.load_state_dict(state_dict)
else:
    state_dict_path = os.path.join(XLMR_MODEL_DIR, "pytorch_model.bin")
    state_dict = torch.load(state_dict_path, map_location="cpu")
    xlmr_model.load_state_dict(state_dict)

xlmr_model.to(device)
xlmr_model.eval()

print("Loaded trained XLM-R large model.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded trained XLM-R large model.


In [ ]:
def predict_token_with_confidence(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    if token_class not in ["EN", "MIX"]:
        return {
            "original_token": token,
            "token_class": token_class,
            "pred_root": None,
            "pred_suffix": None,
            "root_conf": None,
            "suffix_conf": None,
            "corrected_token": token
        }

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)
    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN_XLMR,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(input_ids=input_ids, attention_mask=attention_mask)
        root_probs = F.softmax(outputs["root_logits"], dim=1)
        suffix_probs = F.softmax(outputs["suffix_logits"], dim=1)

        root_pred_id = torch.argmax(root_probs, dim=1).item()
        suffix_pred_id = torch.argmax(suffix_probs, dim=1).item()

        root_conf = root_probs[0, root_pred_id].item()
        suffix_conf = suffix_probs[0, suffix_pred_id].item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root
    else:
        corrected_token = f"{pred_root}-{pred_suffix}"

    return {
        "original_token": token,
        "token_class": token_class,
        "wrong_root": wrong_root,
        "wrong_suffix": wrong_suffix,
        "pred_root": pred_root,
        "pred_suffix": pred_suffix,
        "root_conf": root_conf,
        "suffix_conf": suffix_conf,
        "corrected_token": corrected_token
    }

In [ ]:
ROOT_CONF_THRESH_MIX = 0.95
SUFFIX_CONF_THRESH_MIX = 0.90

def should_correct_token(pred_info, suspicious_tokens):
    token_class = pred_info["token_class"]
    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    # Only MIX tokens for this model
    if token_class != "MIX":
        return False

    if corrected_token == original_token:
        return False

    # Must preserve same English root for safer suffix correction
    wrong_root = pred_info.get("wrong_root")
    pred_root = pred_info.get("pred_root")

    if wrong_root != pred_root:
        return False

    if (
        pred_info["root_conf"] >= ROOT_CONF_THRESH_MIX and
        pred_info["suffix_conf"] >= SUFFIX_CONF_THRESH_MIX
    ):
        return True

    return False

In [ ]:
def build_hints_for_sentence(asr_sentence, suspicious_tokens):
    tokens = tokenize(asr_sentence)
    hints = []
    debug_rows = []

    for idx in range(len(tokens)):
        pred_info = predict_token_with_confidence(tokens, idx)
        apply_change = should_correct_token(pred_info, suspicious_tokens)

        if apply_change:
            original_token = pred_info["original_token"]
            corrected_token = pred_info["corrected_token"]
            hints.append(f"{original_token}=>{corrected_token}")

        debug_rows.append({
            "index": idx,
            "original_token": pred_info["original_token"],
            "token_class": pred_info["token_class"],
            "pred_root": pred_info.get("pred_root"),
            "pred_suffix": pred_info.get("pred_suffix"),
            "root_conf": pred_info.get("root_conf"),
            "suffix_conf": pred_info.get("suffix_conf"),
            "corrected_token": pred_info.get("corrected_token"),
            "apply_hint": apply_change
        })

    hint_text = " ; ".join(hints)
    return hint_text, debug_rows

In [ ]:
hint_rows = []
all_debug_rows = []

for idx, row in tqdm(asr_df.iterrows(), total=len(asr_df), desc="Building XLM-R hints"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    error_msg = row["error"]

    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    if has_error:
        hint_text = ""
        debug_rows = []
    else:
        hint_text, debug_rows = build_hints_for_sentence(
        generated_text,
        suspicious_tokens
    )

    hint_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "hint_text": hint_text,
        "error": error_msg
    })

    for d in debug_rows:
        d["row_id"] = row_id
        d["generated_text"] = generated_text
        all_debug_rows.append(d)

hint_df = pd.DataFrame(hint_rows)
hint_df.to_csv(HINT_RESULTS_CSV, index=False)

debug_df = pd.DataFrame(all_debug_rows)
debug_df.to_csv(DEBUG_HINTS_CSV, index=False)

print("Saved hint CSV:", HINT_RESULTS_CSV)
print("Saved debug hints CSV:", DEBUG_HINTS_CSV)
hint_df.head()

Building XLM-R hints:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved hint CSV: /content/drive/MyDrive/hint_pipeline_with_hints_1000.csv
Saved debug hints CSV: /content/drive/MyDrive/hint_pipeline_debug_hints_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,hint_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,,None


In [ ]:
hint_df["has_hint"] = hint_df["hint_text"].astype(str).str.strip() != ""

print("Rows with hints:", hint_df["has_hint"].sum())
print("Hint rate:", hint_df["has_hint"].mean())

Rows with hints: 37
Hint rate: 0.037


In [ ]:
debug_df["token_class"].value_counts()

,count
token_class,
TA,4758
EN,2839
MIX,727
OTHER,1


In [ ]:
mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL_PATH)

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MT5_MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

mt5_model.to(device)
mt5_model.eval()

print("Loaded merged mT5 model.")

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

Loaded merged mT5 model.


In [ ]:
def run_mt5_inference_with_hints(asr_text, hint_text, max_input_len=256, max_new_tokens=256):
    try:
        asr_text = str(asr_text).strip()
        hint_text = str(hint_text).strip()

        if hint_text != "":
            prompt = f"fix tamil-english: {asr_text} || hints: {hint_text}"
        else:
            prompt = f"fix tamil-english: {asr_text}"

        inputs = mt5_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_len
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            outputs = mt5_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True
            )

        text = mt5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        return text, None

    except Exception as e:
        return "", str(e)

In [ ]:
test_asr = hint_df.loc[0, "generated_text"]
test_hints = hint_df.loc[0, "hint_text"]

print("ASR   :", test_asr)
print("HINTS :", test_hints)

pred_text, err = run_mt5_inference_with_hints(test_asr, test_hints)

print("OUTPUT:", pred_text)
print("ERROR :", err)

ASR   : Online class join பண்ண late ஆனதால start miss ஆயிட்டு
HINTS : 
OUTPUT: Online class join பண்ண late ஆனதால start miss ஆயிட்டு
ERROR : None


In [ ]:
final_rows = []

for idx, row in tqdm(hint_df.iterrows(), total=len(hint_df), desc="Running mT5 with XLM-R hints"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    hint_text = str(row["hint_text"]).strip()
    error_msg = row["error"]

    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    if has_error:
        mt5_hint_text = ""
        mt5_error = error_msg
    else:
        mt5_hint_text, mt5_error = run_mt5_inference_with_hints(generated_text, hint_text)

    final_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "hint_text": hint_text,
        "mt5_with_hints_text": mt5_hint_text,
        "error": mt5_error
    })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(FINAL_RESULTS_CSV, index=False)

print("Saved final hint-based pipeline results:", FINAL_RESULTS_CSV)
final_df.head()

Running mT5 with XLM-R hints:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved final hint-based pipeline results: /content/drive/MyDrive/hint_pipeline_mt5_results_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,hint_text,mt5_with_hints_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,,Itsy help center site-ல selling on itsy-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [ ]:
hint_df["has_hint"] = hint_df["hint_text"].astype(str).str.strip() != ""
print("Rows with hints:", hint_df["has_hint"].sum())
print("Rows without hints:", (~hint_df["has_hint"]).sum())
print("Hint rate:", hint_df["has_hint"].mean())

Rows with hints: 37
Rows without hints: 963
Hint rate: 0.037


In [ ]:
non_empty_hints = hint_df[hint_df["hint_text"].astype(str).str.strip() != ""]
non_empty_hints[["generated_text", "hint_text"]].head(20)

,generated_text,hint_text
17,அவன் படிப்புக்குர teacher-ஏ மதிக்க மாட்டான்,teacher-ஏ=>teacher-அயே
32,நான் உங்களுக்கு money dance-அ பண்ணான் ஆனா serv...,issue-ஆ=>issue-வா
33,எனக்கு n global transit எண்டு வருது bro என்ன p...,problem-ஆ=>problem-அ
96,அண்ணா இது எந்த company-இடம் தெரியுமா இலங்கையில்,company-இடம்=>company-களும்
110,Bro இந்த smartphone எல்லா store-லயும் இருக்கும...,store-லயும்=>store-லையும்
134,Ebay first time purchase பண்ண போறன் அது safe-ஆ...,safe-ஆ=>safe-அ
176,அப்பா phone-அ தட்டி விட்டார்,phone-அ=>phone-ஐ
204,Bro நான் try பண்ணி பாத்தப்ப payment free-யா ஆகுது,free-யா=>free-ஆ
261,Temu app safe-ஆ bro order பண்ணலாமா,safe-ஆ=>safe-அ
293,அவன் படிப்பிக்கிற teacher-ஐயே மதிக்க மாட்டான்,teacher-ஐயே=>teacher-அயே


In [ ]:
valid_final_df = final_df[final_df["error"].isna() | (final_df["error"] == "")].copy()

expected_list = valid_final_df["expected_text"].astype(str).tolist()
asr_list = valid_final_df["generated_text"].astype(str).tolist()
hint_mt5_list = valid_final_df["mt5_with_hints_text"].astype(str).tolist()

metrics = {
    "num_rows_total": int(len(final_df)),
    "num_rows_valid": int(len(valid_final_df)),
    "asr_wer": float(wer(expected_list, asr_list)),
    "asr_cer": float(cer(expected_list, asr_list)),
    "hint_mt5_wer": float(wer(expected_list, hint_mt5_list)),
    "hint_mt5_cer": float(cer(expected_list, hint_mt5_list)),
}

print(json.dumps(metrics, indent=2, ensure_ascii=False))

with open(FINAL_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved metrics:", FINAL_METRICS_JSON)

{
  "num_rows_total": 1000,
  "num_rows_valid": 1000,
  "asr_wer": 0.1361609832509941,
  "asr_cer": 0.041790079324549756,
  "hint_mt5_wer": 0.14519821665260874,
  "hint_mt5_cer": 0.05666768801280087
}
Saved metrics: /content/drive/MyDrive/hint_pipeline_metrics_1000.json


In [ ]:
comparison = {
    "ASR": {
        "WER": float(wer(expected_list, asr_list)),
        "CER": float(cer(expected_list, asr_list)),
    },
    "ASR_HINT_mT5": {
        "WER": float(wer(expected_list, hint_mt5_list)),
        "CER": float(cer(expected_list, hint_mt5_list)),
    }
}

print(json.dumps(comparison, indent=2, ensure_ascii=False))

{
  "ASR": {
    "WER": 0.1361609832509941,
    "CER": 0.041790079324549756
  },
  "ASR_HINT_mT5": {
    "WER": 0.13977587661163995,
    "CER": 0.05002893814046914
  }
}
